In [ ]:
# @title Run Daily Options Screener (Final Verified Version + Options Date Filter)
# 1. INSTALL LIBRARIES
!pip install yfinance pandas_ta -q

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta

# 2. DEFINE THE TICKER LIST
raw_tickers = """
A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM
"""
ticker_list = [t.strip() for t in raw_tickers.split() if t.strip() and t.strip() != "XYZ"]

# 3. HELPER FUNCTIONS: EARNINGS, OPTIONS & CORRELATION
def has_earnings_coming_up(ticker_symbol, days=15):
    """
    Checks yfinance calendar for earnings in the next X days.
    """
    try:
        t = yf.Ticker(ticker_symbol)
        cal = t.calendar
        earnings_dates = []
        
        if cal is None: return False
        elif isinstance(cal, dict):
            keys = cal.keys()
            for k in keys:
                if 'Earnings' in str(k):
                    earnings_dates = cal[k]
                    break
        
        if not earnings_dates: return False

        for date in earnings_dates:
             if hasattr(date, 'tzinfo') and date.tzinfo:
                 date = date.tz_localize(None)
             now = datetime.now()
             delta = date - now
             if 0 <= delta.days <= days:
                 return True
        return False
    except: return False

def has_near_term_options(ticker_symbol, days=12):
    """
    Checks if the ticker has option expirations within the next X days.
    """
    try:
        t = yf.Ticker(ticker_symbol)
        expirations = t.options # Returns a tuple of strings 'YYYY-MM-DD'
        
        if not expirations:
            return False

        now = datetime.now()
        # Create a limit date X days into the future
        limit_date = now + timedelta(days=days)

        for exp_str in expirations:
            # Parse the YYYY-MM-DD string
            exp_date = datetime.strptime(exp_str, '%Y-%m-%d')
            
            # Check if this expiration is between Today and Today+12
            # We assume 'exp_date' is 00:00:00, so we compare purely on dates usually
            if now <= exp_date <= limit_date:
                return True
                
        return False
    except:
        return False

def filter_correlated_assets(candidate_list, price_history_dict, max_items=10, threshold=0.85):
    """
    Selects top assets by volume, skipping any that are highly correlated (>0.85)
    to assets already selected.
    """
    if not candidate_list: return []
    
    # Sort candidates by Dollar Volume (Highest to Lowest)
    sorted_candidates = sorted(candidate_list, key=lambda x: x['DollarVolume'], reverse=True)
    selected = []
    
    for cand in sorted_candidates:
        if len(selected) >= max_items: break
        
        ticker = cand['Ticker']
        
        # If it's the first one, take it
        if not selected:
            selected.append(cand)
            continue
            
        # Check correlation against already selected assets
        is_duplicate = False
        try:
            # Get last 60 days of returns for current candidate
            cand_series = price_history_dict[ticker].pct_change().tail(60) 
            
            for sel in selected:
                sel_ticker = sel['Ticker']
                sel_series = price_history_dict[sel_ticker].pct_change().tail(60)
                
                # Calculate correlation
                corr = cand_series.corr(sel_series)
                
                if corr > threshold:
                    # Correlation too high
                    is_duplicate = True
                    break
        except:
            is_duplicate = False # If data missing for corr calc, let it pass
        
        if not is_duplicate:
            selected.append(cand)
            
    return selected

# 4. DOWNLOAD & PROCESS (Individual Loop for Accuracy)
print(f"Starting Precision Scan for {len(ticker_list)} tickers...")
print("Settings: Raw Prices, Strict Trend, Options available < 12 Days")

candidates = []
candidate_price_history = {} # Store history here for correlation check later

for i, ticker in enumerate(ticker_list):
    try:
        # Download 1 Year of data to ensure SMA50 is fully calculated
        # auto_adjust=False ensures we get the RAW PRICE (Matches TradingView)
        df = yf.download(ticker, period="1y", auto_adjust=False, progress=False, threads=False)
        
        # Cleanup
        if df.empty or len(df) < 60: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        
        # Sort by date ensures we grab the actual latest candle
        df = df.sort_index()
        
        # Calculate Indicators
        df['SMA_10'] = ta.sma(df['Close'], length=10)
        df['SMA_50'] = ta.sma(df['Close'], length=50)
        df['RSI'] = ta.rsi(df['Close'], length=14)
        
        # Liquidity Check (approx > $25M daily volume to filter junk early)
        avg_price = df['Close'].tail(20).mean()
        avg_vol = df['Volume'].tail(20).mean()
        dollar_vol = avg_price * avg_vol
        
        if dollar_vol < 25000000: continue 

        # Get Latest Candle
        current = df.iloc[-1]
        current_date = df.index[-1].strftime('%Y-%m-%d')
        
        signal = None
        
        # --- BULLISH CRITERIA ---
        # 1. Trend Alignment: SMA 10 MUST be > SMA 50
        # 2. Price Strength: Price MUST be > SMA 10 (and naturally > SMA 50)
        # 3. RSI: 40-65
        if (current['SMA_10'] > current['SMA_50']) and \
           (current['Close'] > current['SMA_10']) and \
           (current['Close'] > current['SMA_50']) and \
           (40 < current['RSI'] < 65):
            signal = "BULLISH"

        # --- BEARISH CRITERIA ---
        # 1. Trend Alignment: SMA 10 MUST be < SMA 50
        # 2. Price Weakness: Price MUST be < SMA 10
        # 3. RSI: 35-60
        elif (current['SMA_10'] < current['SMA_50']) and \
             (current['Close'] < current['SMA_10']) and \
             (current['Close'] < current['SMA_50']) and \
             (35 < current['RSI'] < 60):
            signal = "BEARISH"
            
        if signal:
            # FILTERS: 
            # 1. No Earnings in next 15 days
            # 2. MUST have options expiring in next 12 days
            if not has_earnings_coming_up(ticker, days=15):
                if has_near_term_options(ticker, days=12):
                    candidates.append({
                        'Ticker': ticker,
                        'Signal': signal,
                        'Date': current_date,
                        'DollarVolume': dollar_vol,
                        'Close': round(current['Close'], 2),
                        'SMA_10': round(current['SMA_10'], 2),
                        'SMA_50': round(current['SMA_50'], 2),
                        'RSI': round(current['RSI'], 2)
                    })
                    # Save price history for the Correlation function
                    candidate_price_history[ticker] = df['Close']
                
    except Exception as e:
        continue
    
    # Progress indicator
    if i % 50 == 0: print(f"Scanned {i}/{len(ticker_list)} tickers...")

# 5. POST-PROCESSING (Correlation & Display)
if candidates:
    bull_raw = [c for c in candidates if c['Signal'] == 'BULLISH']
    bear_raw = [c for c in candidates if c['Signal'] == 'BEARISH']
    
    print(f"Processing Correlation Filters on {len(candidates)} candidates...")
    final_bulls = filter_correlated_assets(bull_raw, candidate_price_history, max_items=10)
    final_bears = filter_correlated_assets(bear_raw, candidate_price_history, max_items=10)
    
    bulls_df = pd.DataFrame(final_bulls)
    bears_df = pd.DataFrame(final_bears)
    
    print("\n" + "="*80)
    print("TRADINGVIEW WATCHLIST STRINGS (Uncorrelated, Raw Data)")
    print("="*80)
    
    if not bulls_df.empty:
        print(f"\n>>> BULLISH WATCHLIST (Credit Puts):")
        print(", ".join([x['Ticker'] for x in final_bulls]))
    else:
        print("\n>>> BULLISH WATCHLIST: None found.")
    
    if not bears_df.empty:
        print(f"\n>>> BEARISH WATCHLIST (Credit Calls):")
        print(", ".join([x['Ticker'] for x in final_bears]))
    else:
        print("\n>>> BEARISH WATCHLIST: None found.")
    
    print("\n" + "="*80)
    print("DETAILED ANALYSIS")
    print("="*80)
    
    display_cols = ['Ticker', 'Date', 'Close', 'SMA_10', 'SMA_50', 'RSI']
    
    if not bulls_df.empty:
        print("\n--- BULLISH (SMA10 > SMA50 & Price > Both) ---")
        print(bulls_df[display_cols].to_string(index=False))
        
    if not bears_df.empty:
        print("\n--- BEARISH (SMA10 < SMA50 & Price < Both) ---")
        print(bears_df[display_cols].to_string(index=False))

else:
    print("No trades found matching criteria today.")

# 6. PRINT RULES
print("\n" + "="*80)
print("⚠️ TRADE MANAGEMENT RULES")
print("="*80)
print("1. ENTRY:")
print("   - Expiration: ~9 Days to Expiration (DTE).")
print("   - Safety: No Earnings within 15 Days. Options must exist < 12 Days.")
print("   - Strike: Sell strike with ~$0.20 - $0.25 credit.Take profit at 75%")
print("   - Spread Width: $1.00 (or $2.50/5.00 for larger stocks).")
print("\n2. STOP LOSS (THE EDGE):")
print("   - Set price alert at SOLD STRIKE.")
print("   - IF Stock Price CROSSES/CLOSES past Sold Strike -> CLOSE THE TRADE.")
print("="*80)